# 03 - Second tour brut, duel par duel

Même logique que pour le premier tour : une seule hypothèse à la fois. Aucun mélange entre duels concurrents.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

try:
    from IPython.display import display
except ImportError:
    display = print

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
pio.renderers.default = "json"

from presidentielle2027.analytics.trends import smooth_candidate_trends
from presidentielle2027.dashboard.colors import get_political_color
from presidentielle2027.extraction.canonicalization import canonicalize_candidate_fields

path = REPO_ROOT / "data" / "processed" / "wikipedia_2027_polls_normalized_v2.csv"
frame = pd.read_csv(path)
canonical = frame.apply(lambda row: canonicalize_candidate_fields(row.get("candidate_name"), row.get("candidate_party"), row.get("political_family")), axis=1, result_type="expand")
canonical.columns = ["candidate_name", "candidate_party", "political_family"]
frame[["candidate_name", "candidate_party", "political_family"]] = canonical
frame["publication_date"] = pd.to_datetime(frame["publication_date"], errors="coerce")
frame["estimate_percent"] = pd.to_numeric(frame["estimate_percent"], errors="coerce")
frame["sample_size"] = pd.to_numeric(frame.get("sample_size"), errors="coerce")
frame = frame.loc[frame["round"] == "second_round"].copy()
matchup_catalog = frame.groupby("scenario_name", dropna=False).agg(rows=("poll_id", "count"), pollsters=("polling_company", "nunique"), candidates=("candidate_name", "nunique"), last_publication=("publication_date", "max")).sort_values(["last_publication", "rows"], ascending=[False, False])
display(matchup_catalog)

In [ ]:
MATCHUP_NAME = matchup_catalog.index[0]
matchup_frame = frame.loc[frame["scenario_name"] == MATCHUP_NAME].copy()
matchup_frame = smooth_candidate_trends(matchup_frame)
pd.Series({
    "matchup": MATCHUP_NAME,
    "rows": len(matchup_frame),
    "pollsters": matchup_frame["polling_company"].nunique(),
    "candidates": matchup_frame["candidate_name"].nunique(),
    "first_publication": matchup_frame["publication_date"].min(),
    "last_publication": matchup_frame["publication_date"].max(),
})

In [ ]:
display(matchup_frame[["publication_date", "polling_company", "candidate_name", "estimate_percent", "sample_size", "source_url"]].sort_values(["publication_date", "estimate_percent"], ascending=[False, False]).head(40))

pollster_matrix = matchup_frame.pivot_table(index="candidate_name", columns="polling_company", values="estimate_percent", aggfunc="mean")
pollster_matrix

In [ ]:
figure = go.Figure()
for candidate_name, group in matchup_frame.groupby("candidate_name", dropna=False):
    ordered = group.sort_values("publication_date")
    party = ordered["candidate_party"].dropna().iloc[0] if ordered["candidate_party"].notna().any() else None
    family = ordered["political_family"].dropna().iloc[0] if ordered["political_family"].notna().any() else None
    color = get_political_color(party, family)
    figure.add_trace(go.Scatter(x=ordered["publication_date"], y=ordered["estimate_percent"], mode="markers", marker={"size": 8, "color": color, "opacity": 0.7}, name=f"{candidate_name} - points", legendgroup=candidate_name, showlegend=False))
    figure.add_trace(go.Scatter(x=ordered["publication_date"], y=ordered["smoothed_estimate"], mode="lines", line={"width": 2.8, "color": color}, name=candidate_name, legendgroup=candidate_name, showlegend=True))

figure.update_layout(title=f"Second tour brut - {MATCHUP_NAME}", xaxis_title="Date de publication", yaxis_title="Intentions de vote (%)", paper_bgcolor="white", plot_bgcolor="white", legend_title="Candidats")
figure.update_xaxes(showgrid=True, gridcolor="#e5e5e5")
figure.update_yaxes(showgrid=True, gridcolor="#e5e5e5")
figure